# Задание 1

(**NB.** для запуска примеров кода нужен Python версии не ниже **3.10**).

Есть следующий код для [автоматического дифференцирования](https://en.wikipedia.org/wiki/Automatic_differentiation), в котором используются особенности системы типов языка `Python`: 

In [6]:
from dataclasses import dataclass
from typing import Union, Callable
from numbers import Number

@dataclass
class Dual:
    value: float
    d: float

    def __add__(self, other: Union["Dual", Number]) -> "Dual":
         match other:
            case Dual(o_value, o_d):
                return Dual(self.value + o_value, self.d + o_d)
            case Number():
                return Dual(float(other) + self.value, self.d)

    def __mul__(self, other: Union["Dual", Number]) -> "Dual":
         match other:
            case Dual(o_value, o_d):
                return Dual(self.value * o_value, self.value * o_d + self.d * o_value)
            case Number():
                return Dual(float(other) * self.value, float(other) * self.d)    

    __rmul__ = __mul__  # https://docs.python.org/3/reference/datamodel.html#object.__mul__
    __radd__ = __add__  # https://docs.python.org/3/reference/datamodel.html#object.__radd__
 

def diff(func: Callable[[float], float]) -> Callable[[float], float]:
    return lambda x: func(Dual(x, 1.0)).d 

Поддерживаются две операции - сложение и умножение. Применить можно так:

In [7]:
# Функция, которую будем дифференцировать
def f(x: float) -> float:
    return 5 * x * x + 2 * x + 2

f_diff = diff(f)

# значение производной в точке x = 2
f_diff(2)

22.0

## Задание 1.1 (5 баллов)

Какие недостатки вы видите в данной реализации? Реализуйте поддержку (полностью самостоятельно или модифицируя приведенный код):
- [унарных операций](https://docs.python.org/3/reference/datamodel.html#object.__neg__) 
- деления
- возведения в степень

Каким образом можно проверить корректность решения?  Реализуйте достаточный, по вашему мнению, набор тестов.

In [97]:
from dataclasses import dataclass
from typing import Callable
from numbers import Number
from math import log

@dataclass(slots=True)
class Dual:
    value: float
    d: float

    def __add__(self, other: "Dual" | Number) -> "Dual":
        match other:
            case Dual(o_value, o_d):
                return Dual(self.value + o_value, self.d + o_d)
            case Number():
                return Dual(float(other) + self.value, self.d)

    def __mul__(self, other: "Dual" | Number) -> "Dual":
        match other:
            case Dual(o_value, o_d):
                return Dual(self.value * o_value, self.value * o_d + self.d * o_value)
            case Number():
                other_float = float(other)
                return Dual(other_float * self.value, other_float * self.d)

    def __truediv__(self, other: "Dual" | Number):
        match other:
            case Dual(o_value, o_d):
                if o_value == 0:
                    raise ValueError("Деление на дуальное число с нулевым значением")
                return Dual(self.value / o_value, (self.d * o_value - self.value * o_d) / (o_value ** 2))
            case Number():
                other_float = float(other)
                if other_float == 0:
                    raise ValueError("Деление на ноль")
                return Dual(self.value / other_float, self.d / other_float)

    def __rtruediv__(self, other: "Dual" | Number) -> "Dual":
        match other:
            case Dual():
                return other.__truediv__(self)
            case Number():
                if self.value == 0:
                    raise ValueError("Деление на ноль")
                other_float = float(other)
                return Dual(other_float / self.value, -other_float * self.d / (self.value ** 2))

    def __pow__(self, n: int | float) -> "Dual":
        if n == 0:
            return Dual(1.0, 0.0)
        # Степень n <= 0 при нулевом основании дает математическую неопределенность
        # Степень 0 < n < 1 при нулевом основании дает деление на ноль
        if self.value == 0 and n < 1:
            raise ValueError("Производна не определена")
        return Dual(self.value ** n, self.d * n * self.value ** (n - 1))

    def __rpow__(self, base: int | float) -> "Dual":
        # Для вычисления производной a^x, a должно быть строго больше нуля
        if base <= 0:
            raise ValueError("Основание степени должно быть строго больше нуля")
        return Dual(base ** self.value, self.d * base ** self.value * log(base))

    def __pos__(self) -> "Dual":
        return Dual(+self.value, +self.d)

    def __neg__(self) -> "Dual":
        return Dual(-self.value, -self.d)

    def __abs__(self) -> "Dual":
        if self.value == 0:
            raise ValueError("Производная модуля в нуле не определена")
        sign = 1.0 if self.value > 0 else -1.0
        return Dual(abs(self.value), sign * self.d)

    def __invert__(self) -> "Dual":
        return Dual(self.d, self.value)
        
    def __repr__(self):
        return f"Dual({self.value}, {self.d})"

    __rmul__ = __mul__  # https://docs.python.org/3/reference/datamodel.html#object.__mul__
    __radd__ = __add__  # https://docs.python.org/3/reference/datamodel.html#object.__radd__


def diff(func: Callable[[float], float]) -> Callable[[float], float]:
    return lambda x: func(Dual(x, 1.0)).d 

In [100]:
# ТЕСТЫ
from math import isclose
import pytest


# __add__
res = Dual(1.0, 4.5) + Dual(5.7, 0.5)
assert res.value == 6.7, f"Ожидалось 6.7, получено {res.value}"
assert res.d == 5.0, f"Ожидалось 5, получено {res.d}"

res = Dual(1.0, 4.5) + 5.7
assert res.value == 6.7, f"Ожидалось 6.7, получено {res.value}"
assert res.d == 4.5, f"Ожидалось 4.5, получено {res.d}"

res = 23.3 + Dual(9.9, 0)
assert res.value == 33.2, f"Ожидалось 33.2, получено {res.value}"
assert res.d == 0.0, f"Ожидалось 0.0, получено {res.d}"


# __mul__
res = Dual(1.0, 4.5) * Dual(5.7, 0.5)
assert isclose(res.value, 5.7), f"Ожидалось 5.7, получено {res.value}"
assert isclose(res.d, 26.15), f"Ожидалось 26.15, получено {res.d}"

res = Dual(1.0, 4.5) * 5.0
assert isclose(res.value, 5.0), f"Ожидалось 5.0, получено {res.value}"
assert isclose(res.d, 22.5), f"Ожидалось 22.5, получено {res.d}"

res = 5.0 * Dual(5.0, 3.4)
assert isclose(res.value, 25.0), f"Ожидалось 25.0, получено {res.value}"
assert isclose(res.d, 17.0), f"Ожидалось 17.0, получено {res.d}"


# __truediv__
res = Dual(10.0, 40.0) / Dual(5.0, 8.0)
assert isclose(res.value, 2.0), f"Ожидалось 2.0, получено {res.value}"
assert isclose(res.d, 4.8), f"Ожидалось 4.8, получено {res.d}"

res = Dual(10.0, 40.0) / 8.0
assert isclose(res.value, 1.25), f"Ожидалось 1.25, получено {res.value}"
assert isclose(res.d, 5.0), f"Ожидалось 5.0, получено {res.d}"

res = 20.0 / Dual(5.0, 8.0)
assert isclose(res.value, 4.0), f"Ожидалось 4.0, получено {res.value}"
assert isclose(res.d, -6.4), f"Ожидалось -6.4, получено {res.d}"

with pytest.raises(ValueError):
    Dual(1.0, 1.0) / 0.0

with pytest.raises(ValueError):
    30.0 / Dual(0.0, 3.0)


# __pow__
res = Dual(2.0, 5.0) ** 2
assert res.value == 4.0, f"Ожидалось 4.0, получено {res.value}"
assert res.d == 20.0, f"Ожидалось 20.0, получено {res.d}"

res = 3.0 ** Dual(3.0, 2.0)
assert res.value == 27.0, f"Ожидалось 27.0, получено {res.value}"
assert round(res.d, 3) == 59.325, f"Ожидалось 59.325, получено {res.d}"

with pytest.raises(ValueError):
    0.0 ** Dual(3.0, 2.0)

# __neg__
res = -Dual(1.0, 1.0)
assert res.value == -1.0, f"Ожидалось -1.0, получено {res.value}"
assert res.d == -1.0, f"Ожидалось -1.0, получено {res.d}"

# __abs__
res = abs(Dual(-5.0, 2.0))
assert res.value == 5.0, f"Ожидалось 5.0, получено {res.value}"
assert res.d == -2.0, f"Ожидалось -2.0, получено {res.d}"

with pytest.raises(ValueError):
    abs(Dual(0.0, 5.0))

# __invert__
res = ~Dual(-2.0, 5.0)
assert res.value == 5.0, f"Ожидалось 5.0, получено {res.value}"
assert res.d == -2.0, f"Ожидалось -2.0, получено {res.d}"

## Задание 1.2 (7 баллов)
Придумайте способ и реализуйте поддержку функций:
- `exp()`
- `cos()`
- `sin()`
- `log()`

Добавьте соответствующие тесты

In [4]:
# ваш код

## Задание 1.3 (2 балла)

Воспользуйтесь методами **численного** дифференцирования для "проверки" работы кода на нескольких примерах. Используйте библиотечный метод или реализуйте какой-нибудь метод численного дифференцирования самостоятельно (**+1 балл**)

In [5]:
from scipy.misc import derivative

def f(x: float) -> float:
    return 5 * x * x + 2 * x + 2

derivative(f, 2.)

22.0

In [6]:
# ваш код

## Задание 1.4 (6 баллов)

Необходимо разработать систему автоматического тестирования алгоритма дифференцирования в следующем виде:
- реализовать механизм генерации "случайных функций" (например, что-то вроде такого: $f(x) = x + 5 * x - \cos(20 * \log(12 - 20 * x * x )) - 20 * x$ )
- сгенерировать достаточно большое число функций и сравнить результаты символьного и численного дифференцирования в случайных точках 

Генерацию случайных функций можно осуществить, например, двумя путями. 
1. Генерировать функцию в текстовом виде, зачем использовать встроенную функцию [eval](https://docs.python.org/3/library/functions.html#eval)

```python
func = eval("lambda x: 2 * x + 5")
assert func(42) == 89 
```

2. Использовать стандартный модуль [ast](https://docs.python.org/3/library/ast.html), который позволяет во время выполнения программы манипулировать [Абстрактным Синтаксическим Деревом](https://ru.wikipedia.org/wiki/%D0%90%D0%B1%D1%81%D1%82%D1%80%D0%B0%D0%BA%D1%82%D0%BD%D0%BE%D0%B5_%D1%81%D0%B8%D0%BD%D1%82%D0%B0%D0%BA%D1%81%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5_%D0%B4%D0%B5%D1%80%D0%B5%D0%B2%D0%BE).
Например, выражение 

```python
func = lambda x: 2 * x + 5
```

Можно запрограммировать с помощью кода:

```python

expr = ast.Expression(
    body=ast.Lambda(
        args=ast.arguments(
            args=[
                ast.arg(arg='x')
            ],
            posonlyargs=[],
            kwonlyargs=[],
            kw_defaults=[],
            defaults=[]
        ),
        body=ast.BinOp(
            left=ast.BinOp(
                left=ast.Constant(value=2),
                op=ast.Mult(),
                right=ast.Name(id='x', ctx=ast.Load())
            ),
            op=ast.Add(),
            right=ast.Constant(value=5)
        )
    )
)

ast.fix_missing_locations(expr)

func = eval(compile(expr, filename="", mode="eval"))

assert func(42) == 89
```

При реализации нужно учитывать области допустимых значений функций.

In [ ]:
# ваш код

## Задание 1.5 (7 баллов)

Реализуйте поддержку функций нескольких аргументов. Например

```python
def f(x: float, y: float, z: float) -> float:
    return x * y + z - 5 * y  


f_diff = diff(f)

f_diff(10, 10, 10) # = [10, 5, 1]
```

In [ ]:
# ваш код